# Caso: gasto de hogar en la EPF (base `gastos`)

Ejemplo con matrícula de colegio. A diferencia del vino, este gasto es por hogar (`folio`), no por persona (`n_linea`), así que se usa la base `gastos` en vez de `cantidades`, y se agrupa/une por `folio`.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score

codigos_arancel_colegio = [
    "10.1.1.01.02", "10.1.1.02.02", "10.2.1.01.02", "10.2.1.02.02",
]

personas = pd.read_stata("../../data/epf/base-personas-ix-epf-stata.dta", convert_categoricals=False)
gastos = pd.read_stata(
    "../../data/epf/base-gastos-ix-epf-stata.dta", convert_categoricals=False,
    columns=["folio", "ccif", "macrozona", "gasto"],
)

# mismo filtro de siempre: Gran Santiago y códigos de no respuesta afuera
valores_invalidos = [-99, -88, -77]
epf = personas[
    (personas["macrozona"] == 2)
    & (~personas["edad"].isin(valores_invalidos))
    & (~personas["edue"].isin(valores_invalidos))
    & (personas["ing_disp_hog_hd_ai"] >= 0)
].copy()
epf["ing_pc"] = epf["ing_disp_hog_hd_ai"] / epf["npersonas"]

In [2]:
# el arancel es gasto de HOGAR: se agrega y une por folio, no por persona
gasto_arancel = gastos[gastos["ccif"].isin(codigos_arancel_colegio) & (gastos["macrozona"] == 2)]
gasto_por_hogar = gasto_arancel.groupby("folio")["gasto"].sum().reset_index()
gasto_por_hogar.columns = ["folio", "gasto_arancel"]

epf = epf.merge(gasto_por_hogar, on="folio", how="left")
epf["gasto_arancel"] = epf["gasto_arancel"].fillna(0)
epf["incurre_gasto"] = (epf["gasto_arancel"] > 0).astype(int)

print(f"Hogares con arancel: {len(gasto_por_hogar)}")
print(f"Incurren en el gasto: {epf['incurre_gasto'].sum()} / {len(epf)}")

Hogares con arancel: 732
Incurren en el gasto: 2939 / 17340


In [3]:
epf["grupo_escolaridad"] = pd.cut(
    epf["edue"], bins=[-np.inf, 12, 14, 16, np.inf],
    labels=["Escolar", "Tecnico", "Universitaria", "Postgrado"],
)
epf["sexo_lbl"] = epf["sexo"].map({1: "Hombre", 2: "Mujer"})
epf["log_ing_pc"] = np.log(epf["ing_pc"] + 1)  # el ingreso viene muy sesgado, se ajusta mejor en log

modelo_data = epf.dropna(subset=["edad", "grupo_escolaridad", "sexo_lbl", "log_ing_pc"]).copy()

modelo_logit = smf.logit(
    "incurre_gasto ~ C(sexo_lbl) + edad + C(grupo_escolaridad) + log_ing_pc",
    data=modelo_data,
).fit(disp=0)

modelo_data["prob_predicha"] = modelo_logit.predict(modelo_data)
print(modelo_logit.summary())
print("AUC:", roc_auc_score(modelo_data["incurre_gasto"], modelo_data["prob_predicha"]))

                           Logit Regression Results                           
Dep. Variable:          incurre_gasto   No. Observations:                17340
Model:                          Logit   Df Residuals:                    17333
Method:                           MLE   Df Model:                            6
Date:                Thu, 20 Aug 2026   Pseudo R-squ.:                  0.1090
Time:                        19:05:22   Log-Likelihood:                -7031.0
converged:                       True   LL-Null:                       -7891.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
Intercept                               -12.0378      0.391    -30.782      0.000     -12.804     -11.271
C(sexo_lbl)[T.Mujer]                      0.1472  

Ahora el monto, solo entre quienes gastan. Se recortan outliers en el percentil 1-99 antes de ajustar, igual que con el vino.

In [4]:
tabla_gasto = epf[epf["gasto_arancel"] > 0].copy()
tabla_gasto["log_gasto_arancel"] = np.log(tabla_gasto["gasto_arancel"] + 1)

q_ing = tabla_gasto["ing_pc"].quantile([0.01, 0.99])
q_gasto = tabla_gasto["gasto_arancel"].quantile([0.01, 0.99])
tabla_gasto = tabla_gasto[
    tabla_gasto["ing_pc"].between(q_ing.iloc[0], q_ing.iloc[1])
    & tabla_gasto["gasto_arancel"].between(q_gasto.iloc[0], q_gasto.iloc[1])
]

modelo_lineal = smf.ols(
    "log_gasto_arancel ~ C(sexo_lbl) + edad + C(grupo_escolaridad) + log_ing_pc",
    data=tabla_gasto,
).fit()

print(f"n = {len(tabla_gasto)}")
print(modelo_lineal.summary())

n = 2834
                            OLS Regression Results                            
Dep. Variable:      log_gasto_arancel   R-squared:                       0.426
Model:                            OLS   Adj. R-squared:                  0.425
Method:                 Least Squares   F-statistic:                     349.4
Date:                Thu, 20 Aug 2026   Prob (F-statistic):               0.00
Time:                        19:05:22   Log-Likelihood:                -3770.6
No. Observations:                2834   AIC:                             7555.
Df Residuals:                    2827   BIC:                             7597.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------